# SC4001 Phase 5c: hand the network the signal it cannot learn

Accelerator **GPU (T4 x2 or P100)**. No internet needed: code, tables, ESM-2 embeddings and the
precomputed k-mer offsets all come from `isaacwjk21/sc4001-project`.

## The question

Everything measured so far says the same thing. A zero-parameter k-mer nearest-neighbour
baseline beats the DeepDTA CNN on **7 of 8 cold-target partitions**. A frozen ESM-2 branch
improves fit but not ordering, at every projection width from 96 to the full 1280. The network
cannot extract target similarity from roughly 400 protein sequences, and widening the pipe does
not change that.

So stop asking it to. This kernel supplies the similarity signal as an **offset** and lets the
network learn only the correction:

```
prediction = kmer_neighbour_prediction + network(drug, protein)
```

> **RQ.** Does supplying measurement-based target similarity as an input recover cold-target
> performance that a learned protein encoder cannot reach on its own?

Each parent is missing what the other has. The lookup has no drug-side modelling and cannot
leave a drug's observed score range; the network has no usable protein reading. If they compose,
the hybrid beats both.

## The leakage trap, and what was done about it

A training pair's offset must not come from training data containing that pair. It does by
default, and badly: with the tuned settings the k-mer predictor nearly reproduces its own
training scores, **in-sample MSE 0.0216 on DAVIS and 0.0005 on KIBA**. A network trained on
those residuals would learn to correct an error roughly four times smaller than the one it meets
at test time, and the result would look excellent and mean nothing.

`src/kmer_offsets.py` therefore gives training pairs **out-of-fold** offsets: target groups
inside the training set are split into 5 folds and each fold is predicted from the other four,
reproducing the cold-target condition. Measured offset MSE:

| partition | DAVIS | KIBA |
| --- | --- | --- |
| train, in-sample (**not used**) | 0.0216 | 0.0005 |
| train, out-of-fold (used) | 0.3999 | 0.3531 |
| val | 0.3088 | 0.3423 |
| test | 0.3516 | 0.2788 |

Out-of-fold training error sits beside the test error, which is the condition for this
experiment to mean anything. It is slightly *harder* than test on both datasets, since a fold
fits on 4/5 of the training targets, and that is the safe direction to err.

## Arms, each against a width-matched control

| arm | control already measured | control test MSE | control within-ligand CI |
| --- | --- | --- | --- |
| DAVIS, ESM proj 96, residual | `esm/davis` | 0.4021 | 0.6867 |
| DAVIS, ESM proj 1280, residual | `esm-proj1280/davis` | 0.3823 | 0.7002 |
| KIBA, ESM proj 96, residual | `esm/kiba` | 0.3113 | 0.6447 |

DAVIS gets both the matched-width arm and the best neural arm. KIBA stays at 96 because no KIBA
run exists at 1280 and adding one would cost more GPU than the answer is worth.

## Pre-registered success criteria

The hybrid must beat **both** parents, not just the weaker one:

* **DAVIS test MSE below 0.3516** (k-mer alone) and below 0.3823 (best ESM arm).
* **DAVIS within-ligand CI above 0.7494** (k-mer alone), all three seeds in the same direction.
* **KIBA test MSE below 0.2788** (k-mer alone) and below 0.3113 (ESM 96).

Beating only the network means the offset is doing the work and the network is dead weight.
Beating neither means the two signals do not compose, which is itself a reportable result: the
deficit is not merely a missing input.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())
KERNEL_STARTED = time.time()


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# This kernel needs the two modules added for Phase 5 and the extra split columns.
required = ["esm_embed.py", "kmer_baseline.py"]
present = {p.name for p in (project / "src").iterdir()}
missing = [name for name in required if name not in present]
if missing:
    raise SystemExit("Attached dataset predates Phase 5; missing src modules: " + str(missing)
                     + ". Push a new dataset version before running this kernel.")
print("phase 5 modules OK")


In [ ]:
# The frozen ESM-2 embeddings come from the attached dataset: they were computed once by
# sc4001-phase5-esm-coldtarget and uploaded, so nothing is downloaded or recomputed here and
# this kernel needs no internet. src/esm_embed.py is unchanged and still reproduces them.

# Verify every embedding file against its processed target table before training reads it.
import numpy as np
import pandas as pd

processed = project / "data" / "processed"
EXPECTED = {"davis": "davis_targets", "kiba": "kiba_targets", "external": "external_kinases"}
for source, table_stem in EXPECTED.items():
    path = processed / (source + "_esm.npz")
    if not path.exists():
        raise SystemExit("Missing " + str(path))
    with np.load(path, allow_pickle=False) as arrays:
        embeddings = arrays["prot_embeddings"]
    stem = processed / (table_stem + ".csv")
    table = pd.read_csv(stem if stem.exists() else stem.with_suffix(".csv.gz"), usecols=[0])
    if embeddings.shape != (len(table), 1280):
        raise SystemExit(source + " embeddings are " + str(embeddings.shape)
                         + ", expected " + str((len(table), 1280)))
    if not np.isfinite(embeddings).all():
        raise SystemExit(source + " embeddings contain a nonfinite value.")
    norms = np.linalg.norm(embeddings, axis=1)
    duplicates = len(table) - len(np.unique(embeddings, axis=0))
    print("%-9s %s  L2 norm min %.3f mean %.3f max %.3f  duplicate rows %d"
          % (source, embeddings.shape, norms.min(), norms.mean(), norms.max(), duplicates),
          flush=True)
    if duplicates:
        print("   NOTE: identical embeddings mean two targets are indistinguishable to the "
              "protein branch. DAVIS has 423 distinct sequences, so this should be 0.",
              flush=True)


In [ ]:
# Three hybrid arms, each with a width-matched control already measured.
ROOT = "/kaggle/working/outputs"
GRID = [("davis", 96, seed) for seed in (4001, 4002, 4003)]
GRID += [("davis", 1280, seed) for seed in (4001, 4002, 4003)]
GRID += [("kiba", 96, seed) for seed in (4001, 4002, 4003)]
EPOCHS = {"davis": 100, "kiba": 80}

# Measured on this accelerator in earlier kernels: 1.4 min (DAVIS proj96), 1.7 min (DAVIS
# proj1280), 4.6 min (KIBA proj96). Start from the measured figure plus a margin.
PRIOR_SECONDS = {("davis", 96): 110.0, ("davis", 1280): 130.0, ("kiba", 96): 330.0}
BUDGET_SECONDS = 34 * 60

status, skipped = [], []
for dataset, width, seed in GRID:
    epochs = EPOCHS[dataset]
    elapsed = time.time() - KERNEL_STARTED
    done = [row[4] for row in status if row[0] == dataset and row[1] == width and row[3] == 0]
    projected = max(done) if done else PRIOR_SECONDS[(dataset, width)]
    if elapsed + projected > BUDGET_SECONDS:
        skipped.append((dataset, width, seed, round(projected / 60, 1)))
        print("SKIPPING " + dataset + " proj" + str(width) + " seed" + str(seed)
              + ": projected " + format(projected / 60, ".1f") + " min would exceed the "
              + format(BUDGET_SECONDS / 60, ".0f") + " min guard ("
              + format(elapsed / 60, ".1f") + " min used)", flush=True)
        continue
    print()
    print("===== kmer-residual " + dataset + " esm proj" + str(width) + " seed" + str(seed)
          + " (" + str(epochs) + " epochs, no early stop) =====", flush=True)
    t0 = time.time()
    run = subprocess.run(
        [sys.executable, "src/train.py", "--dataset", dataset, "--split", "split_cold_target",
         "--seed", str(seed), "--protein-encoder", "esm", "--esm-proj-dim", str(width),
         "--kmer-residual", "--max-epochs", str(epochs), "--patience", str(epochs), "--resume"],
        cwd=project, env=dict(env, SC4001_OUTPUT_ROOT=ROOT), check=False,
    )
    wall = time.time() - t0
    status.append((dataset, width, seed, run.returncode, wall, epochs))
    print("----- rc=" + str(run.returncode) + " wall=" + format(wall / 60, ".1f") + " min ("
          + format(wall / epochs, ".2f") + " s/epoch) -----", flush=True)

print()
print("Grid wall time", format((time.time() - KERNEL_STARTED) / 60, ".1f"), "min since start")
for dataset, width, seed, rc, wall, epochs in status:
    print("  %-6s proj%-5d seed%d rc=%d %6.1f min  %5.2f s/epoch"
          % (dataset, width, seed, rc, wall / 60, wall / epochs))
failed = [row for row in status if row[3] != 0]
print("FAILED RUNS:", failed if failed else "none")
print("SKIPPED FOR BUDGET:", skipped if skipped else "none")


In [ ]:
# Paired against the width-matched control, and against the k-mer baseline that supplied the
# offset. The hybrid has to beat BOTH parents to mean anything.
import statistics as st

CONTROL = {
    ("davis", 96): {"mse": [0.4194, 0.4011, 0.3857], "ci": [0.8064, 0.8076, 0.8125],
                    "unc_mse": [1.1125, 0.9842, 0.9341], "label": "esm/davis"},
    ("davis", 1280): {"mse": [0.3766, 0.3768, 0.3935], "ci": [0.8117, 0.8175, 0.8110],
                      "unc_mse": [0.9577, 0.9387, 1.0146], "label": "esm-proj1280/davis"},
    ("kiba", 96): {"mse": [0.3123, 0.3070, 0.3148], "ci": [0.7729, 0.7677, 0.7582],
                   "label": "esm/kiba"},
}
KMER = {"davis": {"mse": 0.3516, "ci": 0.8385, "unc_mse": 0.6875, "within": 0.7494},
        "kiba": {"mse": 0.2788, "ci": 0.7871, "within": 0.6467}}
CNN = {"davis": {"mse": 0.3991, "ci": 0.7885}, "kiba": {"mse": 0.3501, "ci": 0.7715}}

runs = {}
for path in sorted(Path(ROOT).glob("kmer-residual-*/**/metrics.json")):
    payload = json.loads(path.read_text())
    saved = json.loads((path.parent / "config.json").read_text())
    runs.setdefault((payload["dataset"], int(saved["esm_proj_dim"])), []).append(payload)

print("%-7s %-6s %-4s %-12s %-12s %-12s" % ("dataset", "width", "n", "test MSE", "test CI",
                                            "uncens MSE"))
hybrid = {}
for key in sorted(runs):
    payloads = sorted(runs[key], key=lambda row: row["seed"])
    hybrid[key] = {
        "mse": [p["test"]["all"]["mse"] for p in payloads],
        "ci": [p["test"]["all"]["ci"] for p in payloads],
        "unc_mse": [p["test"]["uncensored"]["mse"] for p in payloads
                    if "uncensored" in p["test"]],
        "seeds": [p["seed"] for p in payloads],
    }
    row = hybrid[key]
    unc = st.mean(row["unc_mse"]) if row["unc_mse"] else float("nan")
    print("%-7s %-6d %-4d %-12.4f %-12.4f %-12.4f"
          % (key[0], key[1], len(row["mse"]), st.mean(row["mse"]), st.mean(row["ci"]), unc))

for key in sorted(hybrid):
    dataset, width = key
    row, control = hybrid[key], CONTROL.get(key)
    if control is None or len(row["mse"]) != 3:
        print()
        print("=== %s proj%d: %d of 3 seeds, no paired test ===" % (dataset, width, len(row["mse"])))
        continue
    print()
    print("===== %s proj%d hybrid vs its control (%s), paired by seed, 2 df ====="
          % (dataset, width, control["label"]))
    for label, key_name in (("test MSE", "mse"), ("test CI", "ci"), ("uncens MSE", "unc_mse")):
        if key_name not in control or not row.get(key_name):
            continue
        diffs = [a - b for a, b in zip(row[key_name], control[key_name])]
        mean, sd = st.mean(diffs), st.stdev(diffs)
        t = mean / (sd / len(diffs) ** 0.5) if sd > 0 else float("inf")
        signs = " ".join("+" if d > 0 else "-" for d in diffs)
        print("  %-11s %+.4f  t(2)=%+6.2f  seed signs %s" % (label, mean, t, signs))
    print("  Against the parents, on the mean of %d seeds:" % len(row["mse"]))
    print("    test MSE  hybrid %.4f | kmer %.4f | control %.4f | cnn %.4f"
          % (st.mean(row["mse"]), KMER[dataset]["mse"], st.mean(control["mse"]),
             CNN[dataset]["mse"]))
    print("    test CI   hybrid %.4f | kmer %.4f | control %.4f | cnn %.4f"
          % (st.mean(row["ci"]), KMER[dataset]["ci"], st.mean(control["ci"]),
             CNN[dataset]["ci"]))
    beats_kmer = st.mean(row["mse"]) < KMER[dataset]["mse"]
    beats_control = st.mean(row["mse"]) < st.mean(control["mse"])
    print("    beats k-mer on MSE: %s | beats its control: %s"
          % (beats_kmer, beats_control))
    if beats_control and not beats_kmer:
        print("    Reading: the offset is doing the work and the network adds nothing.")
    if beats_kmer and beats_control:
        print("    Reading: the two signals compose. This is the positive result.")
    if not beats_control and not beats_kmer:
        print("    Reading: they do not compose; the deficit is not merely a missing input.")


In [ ]:
# The decisive view, as everywhere else in this project: can it order one unseen target against
# another, inside a single drug's own pairs?
sys.path.insert(0, str(project))
import numpy as np
import pandas as pd
from src.config import Config
from src.data import load_esm_features, processed_csv, resolve_split_column
from src.model import DeepDTA

DATA = project / "data" / "processed"
WITHIN_REF = {"davis": {"cnn": 0.6789, "esm96": 0.6867, "esm1280": 0.7002, "kmer": 0.7494,
                        "measured": 0.6391},
              "kiba": {"cnn": 0.6363, "esm96": 0.6447, "kmer": 0.6467, "measured": 0.5762}}
# The settings src/kmer_baseline.py tuned on validation for each dataset's cold-target split,
# and therefore the settings src/kmer_offsets.py used to build the saved offsets. Hardcoded
# because outputs/kmer/ is not part of the attached dataset.
TUNED = {"davis": {"neighbours": 10 ** 6, "sharpening": 16},
         "kiba": {"neighbours": 20, "sharpening": 4}}
device = torch.device("cuda")


def pairwise_ci(y, prediction):
    """Return one drug's concordance index, prediction ties counted as half."""
    y, prediction = np.asarray(y, dtype=float), np.asarray(prediction, dtype=float)
    better = y[:, None] > y[None, :]
    if not better.any():
        return None
    gap = prediction[:, None] - prediction[None, :]
    return float(((gap[better] > 0).sum() + 0.5 * (gap[better] == 0).sum()) / better.sum())


def batched(model, drug_tokens, protein_features, drug_rows, protein_rows, batch=1024):
    """Predict for aligned drug/target row-index arrays."""
    out = []
    with torch.no_grad():
        for start in range(0, len(drug_rows), batch):
            d = drug_tokens[drug_rows[start:start + batch]].to(device)
            p = protein_features[protein_rows[start:start + batch]].to(device)
            out.append(model(d, p).detach().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

# The offsets the hybrid adds at evaluation time, indexed exactly as src/data.py indexes them.
def load_offsets(dataset, split, n_pairs):
    """Return the per-pair k-mer offset array for this dataset and split."""
    path = DATA / (dataset + "_" + split + "_kmer_offset.npz")
    with np.load(path, allow_pickle=False) as arrays:
        offsets = arrays["offsets"]
    if offsets.shape != (n_pairs,):
        raise SystemExit(str(path) + " has shape " + str(offsets.shape))
    return offsets


summary = {}
for path in sorted(Path(ROOT).glob("kmer-residual-*/**/metrics.json")):
    run_dir = path.parent
    saved = json.loads((run_dir / "config.json").read_text())
    config = Config(**{k: v for k, v in saved.items() if k in Config.__dataclass_fields__})
    dataset, width = config.dataset, config.esm_proj_dim
    model = DeepDTA(saved["vocab_sizes"]["smiles"], saved["vocab_sizes"]["protein"], config).to(device)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=device, weights_only=True))
    model.eval()

    pairs = pd.read_csv(processed_csv(DATA, dataset + "_pairs"),
                        dtype={"ligand_id": str, "target_id": str})
    ligands = pd.read_csv(processed_csv(DATA, dataset + "_ligands"), usecols=["ligand_id"],
                          dtype={"ligand_id": str})
    targets = pd.read_csv(processed_csv(DATA, dataset + "_targets"), usecols=["target_id"],
                          dtype={"target_id": str})
    ligand_row = dict(zip(ligands.ligand_id, range(len(ligands))))
    target_row = dict(zip(targets.target_id, range(len(targets))))
    offsets = load_offsets(dataset, config.split, len(pairs))
    test = pairs.loc[resolve_split_column(pairs, DATA, dataset, config.split) == "test"].copy()
    test["offset"] = offsets[test.index.to_numpy()]

    with np.load(DATA / (dataset + "_tokens.npz"), allow_pickle=False) as arrays:
        drug_tokens = torch.tensor(arrays["drug_tokens"], dtype=torch.long)
    protein_features = load_esm_features(DATA, dataset, len(targets), config.esm_dim)

    test["pred"] = batched(model, drug_tokens, protein_features,
                           test.ligand_id.map(ligand_row).to_numpy(),
                           test.target_id.map(target_row).to_numpy()) + test.offset.to_numpy()
    measured = test.groupby("ligand_id").score.std(ddof=1)
    matched = sorted(measured[(measured > 0) & measured.notna()].index)
    cis = [v for drug in matched
           if (v := pairwise_ci(test.loc[test.ligand_id == drug, "score"],
                                test.loc[test.ligand_id == drug, "pred"])) is not None]

    # The full drug x cold-target grid needs an offset for pairs that were never measured, so
    # the k-mer predictor is reapplied here rather than read from the saved per-pair file.
    sequences = pd.read_csv(processed_csv(DATA, dataset + "_targets"),
                            usecols=["target_id", "sequence"], dtype={"target_id": str})
    from src.kmer_baseline import kmer_similarity, predict_pairs
    similarity = kmer_similarity(sequences.sequence, 3)
    train = pairs.loc[resolve_split_column(pairs, DATA, dataset, config.split) == "train"].copy()
    train["row"] = train.target_id.map(target_row)
    by_drug = {drug: (group.row.to_numpy(), group.score.to_numpy(dtype=np.float64))
               for drug, group in train.groupby("ligand_id")}
    tuned = TUNED[dataset]
    test_targets = sorted(test.target_id.unique())
    grid_drug_rows = np.repeat([ligand_row[d] for d in matched], len(test_targets))
    grid_prot_rows = np.tile([target_row[t] for t in test_targets], len(matched))
    grid_offsets = predict_pairs(np.repeat(matched, len(test_targets)), grid_prot_rows, by_drug,
                                 similarity, float(train.score.mean()), tuned["neighbours"],
                                 tuned["sharpening"])
    grid = batched(model, drug_tokens, protein_features,
                   grid_drug_rows, grid_prot_rows) + grid_offsets
    spreads = grid.reshape(len(matched), len(test_targets)).std(axis=1, ddof=1)
    reference = WITHIN_REF[dataset]

    summary.setdefault((dataset, width), []).append((config.seed, float(np.mean(cis)),
                                                     float(spreads.mean())))
    print("%-6s proj%-5d seed%d  within-ligand CI %.4f  predicted spread %.4f  ratio %.3f"
          % (dataset, width, config.seed, np.mean(cis), spreads.mean(),
             spreads.mean() / reference["measured"]), flush=True)

print()
for key in sorted(summary):
    dataset, width = key
    rows = summary[key]
    ci = sum(r[1] for r in rows) / len(rows)
    spread = sum(r[2] for r in rows) / len(rows)
    reference = WITHIN_REF[dataset]
    print("===== %s proj%d within-ligand, %d seed(s) =====" % (dataset, width, len(rows)))
    print("  hybrid   CI %.4f  spread/measured %.3f" % (ci, spread / reference["measured"]))
    for name in ("kmer", "esm1280", "esm96", "cnn"):
        if name in reference:
            print("  %-8s CI %.4f   (hybrid %+.4f)" % (name, reference[name],
                                                       ci - reference[name]))
    signs = " ".join("+" if r[1] > reference["kmer"] else "-" for r in rows)
    print("  per-seed signs against k-mer alone: %s" % signs)
    print("  PASS requires CI above k-mer's %.4f with all seeds the same sign."
          % reference["kmer"])
